In [3]:
import pandas as pd

df = pd.read_csv("data.csv")

# default column
df["mice"] = False

In [4]:
df

,Sanjay Sir Markings,Category,Contact Name,Designation,Name Of The Company,Address 1,Address 2,City,Pincode,State,...,Contact Person 2.1,Designation 2.1,Email 4.1,Mobile 4.1,Email 1.1,domain,Unnamed: 25,Unnamed: 26,Unnamed: 27,mice
0,NaN,NaN,Mr. Swapan Sil,Partner,23 DEGREES NORTH,"189 A, Santoshpur Avenue,","Behind Central Bank Of India,",Kolkata,700075,West Bengal,...,NaN,NaN,NaN,NaN,info.23dn,gmail.com,NaN,NaN,NaN,False
1,NaN,NaN,Mr Srinjoy Sen,Founder,3S Travel Network Pvt Ltd,16/1 Prince Anwar Shah Road,NaN,Kolkata,West Bengal,700033,...,NaN,NaN,NaN,NaN,srinjoy,3stravelnetwork.com,NaN,NaN,NaN,False
2,1,Leisure,Mr Raja Banik,Director,AAKASH GANGA HOLIDAYS,"BD 119, SALT LAKE, SECTOR I, NEAR BD MARKET, K...",NaN,Kolkata,NaN,West Bengal,...,NaN,NaN,NaN,NaN,salesaakashganga1994,gmail.com,NaN,NaN,NaN,False
3,NaN,NaN,Mr. Aloke Mondal / Moli Mondal,M D,AAKRIT TOURISM,"B/24, East Rajapur Santoshpur Kolkata 75",NaN,Kolkata,700075,NaN,...,NaN,NaN,NaN,NaN,aakrittourism,gmail.com,NaN,NaN,NaN,False
4,1,Skal Association Member,Mr. Sanjeev Mehra / Venessa Mehra,Directors,AARYAN LEISURE & HOLIDAYS PVT LTD,"7/1C, Hazra Road, (Edcon Swastika Building), 1...","Flat 1B,",Kolkata,700026,West Bengal,...,00 91 7044076004,NaN,NaN,NaN,sanjeev.mehra,aaryanholidays.com,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
464,1,Sanjay sir Imp,Mr. Mohammed Zakauddin,Director,THE HOLIDAY HUB (A UNIT OF CHANDANI TRAVELS),"1/1, Camac Street,","Shree Manjari Building, 1st Floor,",Kolkata,700016,West Bengal,...,NaN,NaN,NaN,NaN,zaka,holidayhub.in,NaN,NaN,NaN,False
465,1,Sanjay sir Imp,Mr. Akbar Amin,Director,Travel Zone,"PS Arcade, 11, Sudder Street,","Ground Floor,",Kolkata,NaN,NaN,...,NaN,NaN,NaN,NaN,akbar,travelzone360.com,NaN,NaN,NaN,False
466,1,Sanjay sir Imp,Mr. Sanjeev Churiwal,Director,NIMBUS TOURS & TRAVELS PVT LTD,"8/1, Loudon Street",NaN,Kolkata,700017,West Bengal,...,NaN,NaN,NaN,NaN,shreyanka,nimbustours.in,NaN,NaN,NaN,False
467,1,Sanjay sir Imp,Ms. Punam Arora,Director,South Wind Travel Agents Pvt Ltd,"35, Chittaranjan Ave, Ground Floor",NaN,Kolkata,700012,West Bengal,...,NaN,NaN,NaN,NaN,punam_southwind,yahoo.com,NaN,NaN,NaN,False


In [ ]:
import pandas as pd
import requests
import urllib.parse
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

df = pd.read_csv("data.csv")
df["mice"] = False

HEADERS = {"User-Agent": "Mozilla/5.0"}

EMAIL_PROVIDERS = ["gmail.com", "yahoo.com", "ymail.com", "hotmail.com", "outlook.com"]


def is_email_provider(domain):
    if pd.isna(domain):
        return False
    domain = str(domain).lower()
    return any(p in domain for p in EMAIL_PROVIDERS)


# -----------------------------
# GET ALL INTERNAL LINKS
# -----------------------------
def get_internal_links(base_url, soup):
    links = set()

    for a in soup.find_all("a", href=True):
        href = a["href"]
        full_url = urljoin(base_url, href)

        # keep only same domain
        if urlparse(full_url).netloc == urlparse(base_url).netloc:
            links.add(full_url)

    return list(links)


# -----------------------------
# CHECK PAGE FOR MICE
# -----------------------------
def page_has_mice(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=8)
        if r.status_code != 200:
            return False

        soup = BeautifulSoup(r.text, "html.parser")
        text = soup.get_text(separator=" ").lower()

        return "mice" in text, soup

    except:
        return False, None


# -----------------------------
# CRAWLER (MULTI PAGE)
# -----------------------------
def crawl_domain_for_mice(domain):
    try:
        if not domain.startswith("http"):
            base_url = "https://" + domain
        else:
            base_url = domain

        print(f"    🌐 Opening: {base_url}")

        r = requests.get(base_url, headers=HEADERS, timeout=10)
        if r.status_code != 200:
            return False

        soup = BeautifulSoup(r.text, "html.parser")

        # check homepage
        text = soup.get_text(separator=" ").lower()
        if "mice" in text:
            return True

        # collect internal links
        links = get_internal_links(base_url, soup)
        links = links[:10]  # limit crawl

        print(f"    🔗 Found {len(links)} internal links")

        for idx, link in enumerate(links):
            print(f"      → Crawling [{idx+1}/{len(links)}]: {link}")

            has_mice, _ = page_has_mice(link)
            if has_mice:
                print("      ✅ MICE FOUND on page!")
                return True

        return False

    except Exception as e:
        print(f"    ❌ Error: {e}")
        return False


# -----------------------------
# SEARCH FOR EMAIL DOMAINS
# -----------------------------
def search_and_check_mice(query):
    try:
        url = "https://html.duckduckgo.com/html/?q=" + urllib.parse.quote(query)

        r = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")

        text = soup.get_text(separator=" ").lower()
        return "mice" in text

    except:
        return False


# -----------------------------
# MAIN LOOP
# -----------------------------
total = len(df)

for i, row in df.iterrows():
    company = row.get("Name Of The Company")
    domain = row.get("domain")

    print(f"\n[STEP {i+1}/{total}] {company}")

    found = False

    if is_email_provider(domain):
        print("  📧 Email domain → search mode")

        query = f"{company} MICE meetings incentives conferences exhibitions"
        print(f"  🔍 Query: {query}")

        found = search_and_check_mice(query)

    else:
        print("  🌍 Website domain → crawler mode")

        found = crawl_domain_for_mice(domain)

    df.at[i, "mice"] = found

    print(f"  ✔ RESULT: {found}")


df.to_csv("data_with_mice.csv", index=False)
print("\nDONE ✔ Saved file")


[STEP 1/469] 23 DEGREES NORTH
  📧 Email domain → search mode
  🔍 Query: 23 DEGREES NORTH MICE meetings incentives conferences exhibitions
  ✔ RESULT: False

[STEP 2/469] 3S Travel Network Pvt Ltd
  🌍 Website domain → crawler mode
    🌐 Opening: https://3stravelnetwork.com
    🔗 Found 10 internal links
      → Crawling [1/10]: https://3stravelnetwork.com/tour/cambodia/
      → Crawling [2/10]: https://3stravelnetwork.com/privacy-policy/
      → Crawling [3/10]: https://3stravelnetwork.com/tour/vietnam/
      → Crawling [4/10]: https://3stravelnetwork.com/tour/spain/
      → Crawling [5/10]: https://3stravelnetwork.com/tour/turkey/
      → Crawling [6/10]: https://3stravelnetwork.com/about/
      → Crawling [7/10]: https://3stravelnetwork.com/tours/
      → Crawling [8/10]: https://3stravelnetwork.com/tour/switzerland/
      → Crawling [9/10]: https://3stravelnetwork.com/to_book/spain-portugal/
      → Crawling [10/10]: https://3stravelnetwork.com/cancellation-refund-policy/
  ✔ RESULT: